# Debugging Semantic Kernel Order Intent Classification

This notebook will help us systematically debug the issue with the OrderIntentFlowSK class and its interaction with Azure OpenAI through Semantic Kernel.

In [1]:
# Import required libraries
import os
from pathlib import Path
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.prompt_template import PromptTemplateConfig
from semantic_kernel.functions.kernel_function_decorator import kernel_function
from semantic_kernel.functions.kernel_arguments import KernelArguments
from semantic_kernel.kernel_pydantic import KernelBaseModel


ENDPOINT = "https://t-toluale-0132-resource.openai.azure.com/"
API_KEY = "1BHVs20ZexZ2kRWF4dP7UrK5vt2pTTpcnaXv1qJ5j4hybIdV2fXuJQQJ99BFACHYHv6XJ3w3AAAAACOGhQjE"
DEPLOYMENT_NAME = "gpt-4o"

In [19]:
# Test data
test_message = "I want to change my order."
test_order = {"items": [{"name": "fries", "quantity": 1}]}

# Define a simple prompt template
test_prompt = """
{%
Test prompt for Semantic Kernel with plugin call
%}

User message: {{$user_message}}
Current order: {{$current_order}}

{%#if (OrderUtils.is_order_empty $current_order)%}
Assistant: The order is empty. <no>
{%else%}
Assistant: The order has items. Let's analyze the message:
{{$user_message}}
<yes>
{%if%}
"""


In [7]:
# Define the utility plugin
class OrderUtils:
    @kernel_function(name="is_order_empty", description="Checks if the current order is empty.")
    def is_order_empty(self, current_order: dict) -> bool:
        print(f"is_order_empty called with: {current_order}")  # Debug print
        return not current_order.get("items")

In [20]:
async def test_semantic_kernel():
    # Create kernel and add Azure service
    kernel = Kernel()
    chat_service = AzureChatCompletion(
        deployment_name=DEPLOYMENT_NAME,
        endpoint=ENDPOINT,
        api_key=API_KEY
    )
    kernel.add_service(chat_service)
    
    # Register the utility plugin
    print("Registering OrderUtils plugin...")
    kernel.add_plugin(OrderUtils(), "OrderUtils")
    
    # Create and register the prompt function
    print("Registering prompt function...")
    prompt_config = PromptTemplateConfig(
        template=test_prompt,
        template_format="semantic-kernel"
    )
    kernel.add_function(
        plugin_name="order_plugin",
        function_name="order_intent",
        prompt_template_config=prompt_config
    )
    
    # Prepare variables
    print("Preparing variables...")
    variables = KernelArguments(
        user_message=test_message,
        current_order=test_order
    )
    
    # Test the utility plugin directly
    print("\nTesting utility plugin directly...")
    utils = kernel.plugins["OrderUtils"]
    is_empty = await utils["is_order_empty"].invoke(
    kernel=kernel,
    arguments=KernelArguments(current_order=test_order)
        )
    print(f"Direct plugin test result: {is_empty}")
    
    # Test the prompt function
    print("\nTesting prompt function...")
    try:
        response = await kernel.invoke(
            plugin_name="order_plugin",
            function_name="order_intent",
            arguments=variables
        )
        print(f"Prompt response: {response}")
    except Exception as e:
        print(f"Error in prompt: {str(e)}")
        raise

# Run the test
await test_semantic_kernel()

Registering OrderUtils plugin...
Registering prompt function...
Preparing variables...

Testing utility plugin directly...
is_order_empty called with: {'items': [{'name': 'fries', 'quantity': 1}]}
Direct plugin test result: 

Testing prompt function...
Prompt response: Assistant: The order has items. Let's analyze the message:  
"I want to change my order."  
<yes>  

How would you like to change your order? You can modify the items, update quantities, or add/remove items. Let me know what you'd like!


In [18]:
# Updated test prompt to use simpler syntax
test_prompt = """
{%
Test prompt for Semantic Kernel with plugin call
%}

User message: {{$user_message}}
Current order: {{$current_order}}

{% set isEmpty = OrderUtils.is_order_empty $current_order %}
{% if isEmpty %}
Assistant: The order is empty. <no>
{% else %}
Assistant: The order has items. Let's analyze the message:
{{$user_message}}
<yes>
{% endif %}
"""

async def test_semantic_kernel():
    # Create kernel and add Azure service
    kernel = Kernel()
    chat_service = AzureChatCompletion(
        deployment_name=DEPLOYMENT_NAME,
        endpoint=ENDPOINT,
        api_key=API_KEY
    )
    kernel.add_service(chat_service)
    
    # Register the utility plugin first
    print("Registering OrderUtils plugin...")
    order_utils = OrderUtils()
    kernel.add_plugin(order_utils, "OrderUtils")
    
    # Create and register the prompt function
    print("Registering prompt function...")
    prompt_config = PromptTemplateConfig(
        template=test_prompt,
        template_format="semantic-kernel"  # Changed from handlebars to semantic-kernel
    )
    kernel.add_function(
        plugin_name="order_plugin",
        function_name="order_intent",
        prompt_template_config=prompt_config
    )
    
    # Prepare variables
    print("Preparing variables...")
    variables = KernelArguments(
        user_message=test_message,
        current_order=test_order
    )
    
    # Test the utility plugin directly
    print("\nTesting utility plugin directly...")
    is_empty = await kernel.invoke(
        plugin_name="OrderUtils",
        function_name="is_order_empty",
        arguments=KernelArguments(current_order=test_order)
    )
    print(f"Direct plugin test result: {is_empty}")
    
    # Test the prompt function
    print("\nTesting prompt function...")
    try:
        response = await kernel.invoke(
            plugin_name="order_plugin",
            function_name="order_intent",
            arguments=variables
        )
        print(f"Prompt response: {response}")
    except Exception as e:
        print(f"Error in prompt: {str(e)}")
        import traceback
        traceback.print_exc()

# Run the test
await test_semantic_kernel()

Registering OrderUtils plugin...
Registering prompt function...
Preparing variables...

Testing utility plugin directly...
is_order_empty called with: {'items': [{'name': 'fries', 'quantity': 1}]}
Direct plugin test result: 

Testing prompt function...
Prompt response: It looks like the order isn't empty and contains items. Based on your message, you want to modify your order. Could you please let me know what changes you'd like to make? For example, would you like to add, remove, or change the quantity of an item?
